<a href="https://colab.research.google.com/github/gloria-kambua/rag-t5/blob/dev/ragt5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q sentence-transformers faiss-cpu transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 41.9 MB/s eta 0:00:00


In [2]:
import numpy as np

In [4]:
import torch
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

Running on: cpu


In [6]:
documents = [
    "RAG stands for Retrieval-Augmented Generation. It combines a language model with an external knowledge source so the model can look up facts before answering.",
    "An embedding is a list of numbers that represents the meaning of a piece of text. Texts with similar meaning have embeddings that are close together.",
    "A vector database stores embeddings and lets you quickly find the ones most similar to a query. FAISS, Pinecone, and Chroma are popular choices.",
    "FAISS is an open-source library from Facebook AI for fast similarity search over large collections of vectors. It runs on CPU or GPU.",
    "Chunking is the process of splitting a long document into smaller pieces before embedding them. Good chunking keeps related ideas together.",
    "A hallucination is when a language model confidently states something that is false. RAG reduces hallucinations by grounding answers in retrieved documents.",
    "Cosine similarity measures the angle between two vectors. A value near 1 means the vectors point in nearly the same direction, so the texts are similar in meaning.",
    "The retrieval step takes a user's question, embeds it, and searches the vector database for the most relevant chunks of text.",
    "The generation step feeds the retrieved chunks plus the question into a language model, which writes an answer grounded in that context.",
    "Fine-tuning changes a model's internal weights by training it further on your data. RAG, in contrast, leaves the model unchanged and supplies knowledge at query time.",
    "A large language model (LLM) is a neural network trained on huge amounts of text to predict the next token. Examples include GPT, Claude, and Llama.",
    "Tokenization breaks text into smaller units called tokens before a model can process it. A token is often a word or part of a word.",
    "The main advantage of RAG is that you can update the system's knowledge just by adding documents, with no expensive retraining of the model.",
    "Top-k retrieval means returning the k most similar documents for a query. A common choice is k=3, which balances enough context against too much noise.",
    "Sentence-transformers is a Python library that produces high-quality sentence embeddings using compact models like all-MiniLM-L6-v2.",
]

print(f"We have {len(documents)} documents in our knowledge base.")

We have 15 documents in our knowledge base.


In [7]:
model_name = "bert-base-uncased" # A common, small pre-trained model for demonstration

# Load the tokenizer using AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define a sample sentence
sentence = "Hello, how are you doing today?"

# Tokenize the sentence
# This returns a dictionary with input_ids, token_type_ids, and attention_mask
encoded_input = tokenizer(sentence, return_tensors='pt')

print("Original Sentence:", sentence)
print("Encoded Input (token IDs):")
print(encoded_input['input_ids'])
print("Attention Mask:")
print(encoded_input['attention_mask'])

# Decode the token IDs back to human-readable text
decoded_output = tokenizer.decode(encoded_input['input_ids'][0])

print("\nDecoded Output:", decoded_output)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Original Sentence: Hello, how are you doing today?
Encoded Input (token IDs):
tensor([[ 101, 7592, 1010, 2129, 2024, 2017, 2725, 2651, 1029,  102]])
Attention Mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

Decoded Output: [CLS] hello, how are you doing today? [SEP]


In [8]:
embedder = SentenceTransformer("all-MiniLM-L6-v2", device=device)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
sample = embedder.encode("What is RAG?")
print("Each embedding is a vector of length:", sample.shape[0])

Each embedding is a vector of length: 384


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 70.5 MB/s eta 0:00:00
Running on: cuda
We have 15 documents in our knowledge base.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Paris
QUESTION: What is the difference between RAG and fine-tuning?

ANSWER: Fine-tuning changes a model's internal weights by training it further on your data

--- Retrieved context the model used ---
[0.810] Fine-tuning changes a model's internal weights by training it further on your data. RAG, in contrast, leaves the model unchanged and supplies knowledge at query time.
[0.577] RAG stands for Retrieval-Augmented Generation. It combines a language model with an external knowledge source so the model can look up facts before answering.
[0.538] The main advantage of RAG is that you can update the system's knowledge just by adding documents, with no expensive retraining of the model.
QUESTION: Why does RAG reduce hallucinations, and what is top-k retrieval?

>>> WITHOUT RAG (model guesses from memory):
recollection of a previously learned memory

>>> WITH RAG (model uses retrieved documents):
by grounding answers in retrieved documents


In [10]:
doc_embeddings = embedder.encode(
    documents,
    normalize_embeddings=True,
    show_progress_bar=True,
)

# Shape is (number of documents, 384).
print("Embeddings shape:", doc_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (15, 384)


In [13]:

dimension = doc_embeddings.shape[1]          # 384
index = faiss.IndexFlatIP(dimension)
index.add(doc_embeddings.astype("float32"))  # FAISS wants float32

print("Documents indexed:", index.ntotal)

Documents indexed: 15


In [14]:
def retrieve(query, k=3):
    query_vec = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(query_vec, k)
    # indices[0] holds the row numbers of the top-k documents.
    results = [(documents[i], float(scores[0][rank])) for rank, i in enumerate(indices[0])]
    return results

In [15]:
for doc, score in retrieve("How does RAG avoid making things up?"):
    print(f"[{score:.3f}] {doc}\n")

[0.659] RAG stands for Retrieval-Augmented Generation. It combines a language model with an external knowledge source so the model can look up facts before answering.

[0.580] A hallucination is when a language model confidently states something that is false. RAG reduces hallucinations by grounding answers in retrieved documents.

[0.565] The main advantage of RAG is that you can update the system's knowledge just by adding documents, with no expensive retraining of the model.



In [16]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large").to(device)

def generate(prompt, max_new_tokens=120):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    output = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Quick test with no retrieval yet, just to confirm it loads.
print(generate("Answer briefly: what is the capital of France?", max_new_tokens=20))

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Paris


In [17]:

def rag_answer(question, k=3):
    # 1. RETRIEVE
    retrieved = retrieve(question, k=k)
    context = "\n".join(doc for doc, score in retrieved)

    # 2. BUILD THE PROMPT
    prompt = (
        "Use the context below to answer the question.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )

    # 3. GENERATE
    answer = generate(prompt, max_new_tokens=120)

    return answer, retrieved

In [ ]:
question = "What is the difference between RAG and fine-tuning?"

answer, retrieved = rag_answer(question)

print("QUESTION:", question)
print("\nANSWER:", answer)
print("\n--- Retrieved context the model used ---")
for doc, score in retrieved:
    print(f"[{score:.3f}] {doc}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 70.5 MB/s eta 0:00:00
Running on: cuda
We have 15 documents in our knowledge base.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Paris
QUESTION: What is the difference between RAG and fine-tuning?

ANSWER: Fine-tuning changes a model's internal weights by training it further on your data

--- Retrieved context the model used ---
[0.810] Fine-tuning changes a model's internal weights by training it further on your data. RAG, in contrast, leaves the model unchanged and supplies knowledge at query time.
[0.577] RAG stands for Retrieval-Augmented Generation. It combines a language model with an external knowledge source so the model can look up facts before answering.
[0.538] The main advantage of RAG is that you can update the system's knowledge just by adding documents, with no expensive retraining of the model.
QUESTION: Why does RAG reduce hallucinations, and what is top-k retrieval?

>>> WITHOUT RAG (model guesses from memory):
recollection of a previously learned memory

>>> WITH RAG (model uses retrieved documents):
by grounding answers in retrieved documents


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 70.5 MB/s eta 0:00:00
Running on: cuda
We have 15 documents in our knowledge base.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Paris
QUESTION: What is the difference between RAG and fine-tuning?

ANSWER: Fine-tuning changes a model's internal weights by training it further on your data

--- Retrieved context the model used ---
[0.810] Fine-tuning changes a model's internal weights by training it further on your data. RAG, in contrast, leaves the model unchanged and supplies knowledge at query time.
[0.577] RAG stands for Retrieval-Augmented Generation. It combines a language model with an external knowledge source so the model can look up facts before answering.
[0.538] The main advantage of RAG is that you can update the system's knowledge just by adding documents, with no expensive retraining of the model.
QUESTION: Why does RAG reduce hallucinations, and what is top-k retrieval?

>>> WITHOUT RAG (model guesses from memory):
recollection of a previously learned memory

>>> WITH RAG (model uses retrieved documents):
by grounding answers in retrieved documents
